# CWP Pruning Test: HF ViT CIFAR-10 with sconce
This notebook mirrors the script flow for `nateraw/vit-base-patch16-224-cifar10` and runs Channel-Wise Pruning (CWP) using sconce.

In [ ]:
# Colab/Kaggle/Jupyter-safe setup for Python 3.12
!python -V
!python -m pip install --upgrade pip
!python -m pip install -Uq jedi transformers torchvision torch-pruning snntorch torchprofile prettytable onnxruntime matplotlib

# Clone ViT branch and use source directly (avoids pyproject metadata build issues)
!rm -rf /content/sconce
!git clone -b ViT https://github.com/satabios/sconce.git /content/sconce

import sys
if "/content/sconce" not in sys.path:
    sys.path.insert(0, "/content/sconce")

import sconce as sconce_pkg
print("Using sconce from:", sconce_pkg.__file__)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoModelForImageClassification

from sconce import sconce

MODEL_ID = "nateraw/vit-base-patch16-224-cifar10"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

In [ ]:
from collections import defaultdict, OrderedDict
import copy
import numpy as np

import torch
from torch import nn
from torch.optim import *
from torch.optim.lr_scheduler import *
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
from torchvision.transforms import *
import torch.optim as optim

from transformers import AutoImageProcessor, AutoModelForImageClassification
from sconce import sconce

# Optional hard requirement, similar to your snippet.
REQUIRE_CUDA = False
if REQUIRE_CUDA:
    assert torch.cuda.is_available(), (
        "The current runtime does not have CUDA support. "
        "Please switch runtime to GPU."
    )

processor = AutoImageProcessor.from_pretrained(MODEL_ID)
proc_size = processor.size["height"] if isinstance(processor.size, dict) else processor.size
image_size = int(proc_size)

mean = processor.image_mean
std = processor.image_std

transforms = {
    "train": Compose([
        Resize((image_size, image_size)),
        RandomCrop(image_size, padding=4),
        RandomHorizontalFlip(),
        ToTensor(),
        Normalize(mean=mean, std=std),
    ]),
    "test": Compose([
        Resize((image_size, image_size)),
        ToTensor(),
        Normalize(mean=mean, std=std),
    ]),
}

dataset = {}
for split in ["train", "test"]:
    dataset[split] = CIFAR10(
        root="data/cifar10",
        train=(split == "train"),
        download=True,
        transform=transforms[split],
    )

dataloader = {}
for split in ["train", "test"]:
    dataloader[split] = DataLoader(
        dataset[split],
        batch_size=128*4,
        shuffle=(split == "train"),
        num_workers=0,
        pin_memory=True,
    )

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = AutoModelForImageClassification.from_pretrained(MODEL_ID)

    def forward(self, x):
        # sconce expects tensor outputs for loss/eval paths.
        return self.model(pixel_values=x).logits

model = Net().to(device)



# Define all parameters (README style)
sconces_full = sconce()
sconces_full.model = model  # Model Definition
sconces_full.criterion = nn.CrossEntropyLoss()  # Loss
sconces_full.optimizer = optim.Adam(sconces_full.model.parameters(), lr=1e-4)
sconces_full.scheduler = optim.lr_scheduler.CosineAnnealingLR(sconces_full.optimizer, T_max=200)
sconces_full.dataloader = dataloader
sconces_full.epochs = 5  # Number of epochs
sconces_full.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sconces_full.experiment_name = "vit-cwp-full"  # Define your experiment name here
sconces_full.prune_mode = "CWP"
sconces_full.attention_heads = True

sconces_full.compress()
